# Frame inspector: every validation frame, every model, and why each box was right or wrong

**Inputs (attach both as Kaggle datasets):**
1. **DSERT-RoLL** as you already have it: `processed_data/<Weather>/<drive>/` with `label.pkl`, the LiDAR and radar
   `.npy` files and the camera images. It may be split over several datasets; the notebook finds the pieces.
2. **The four `result.pkl` files** from Snellius. Every evaluation already ran its model on every validation frame and
   saved all predicted boxes there; nothing is run again here. Download them (OnDemand file browser) from
   `~/DSERT-RoLL-3DOD/detection/output/det_model_cfgs/dsert/lidar_only/` and name them after the model:

   | upload as | from |
   |---|---|
   | `radar_scratch.pkl` | `radar_scratch/eval/eval_with_train/epoch_20/val/result.pkl` |
   | `radar_ft_from_livox.pkl` | `radar_ft_from_livox/eval/eval_with_train/epoch_20/val/result.pkl` |
   | `livox_e20_on_radar.pkl` | `livox_e20_on_radar/eval/epoch_20/val/result.pkl` |
   | `livox_pretrain.pkl` | `livox_pretrain/eval/eval_with_train/epoch_20/val/result.pkl` |

   Optional: the `log_train_*.txt` / `log_eval_*.txt` files of those runs, to compare with the official AP.

On a laptop (no `/kaggle/input`) it looks in `~/dsert-roll-sample` and `dsert_results/` instead (see `INPUTS` in section 1).

Run the cells top to bottom. The first run reads every radar and LiDAR file once (section 1) and caches the result in
the output folder; after that everything is fast.

## How the scores work

**The official number (AP, the "0.36" on wandb)** is computed over *all* frames at once, per class:
1. Take every box the model predicted in the whole validation set, sorted by its score (the model's confidence).
2. Go down the list. A box counts as **correct** if it overlaps a labelled box of the same class enough (3D IoU
   ≥ 0.5 for vehicles, ≥ 0.3 for pedestrians and bikes, within 70 m) and that labelled box wasn't taken by a
   higher-scored box. Otherwise it is a **false box**.
3. After each box: *precision* = correct / boxes so far, *recall* = correct / all labelled boxes.
4. **AP = area under the precision-recall curve**: one number for the whole set. There is no "AP of a frame".

**Per frame (this notebook)** we fix one score cut-off per model (the one with the best balance of precision and
recall over the validation set, printed in section 2) and keep the boxes above it. Then for each frame:
* **found** = labelled boxes matched by a kept box, **missed** = labelled boxes nobody matched,
  **false** = kept boxes that match no labelled box
* verdict: **GOOD** = everything found and no false boxes, **BAD** = at least one miss,
  **UGLY** = at least one false box (a frame can be BAD+UGLY)

Every miss and every false box gets a **reason** (section 4), and section 5 draws any frame with all of it written
on the picture.

## 1. Load

In [ ]:
import os, io, re, zipfile, pickle, time, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

ALL_MODELS = ["radar_scratch", "radar_ft_from_livox", "livox_e20_on_radar", "livox_pretrain"]
NAME = {"radar_scratch": "radar, from scratch", "radar_ft_from_livox": "radar, fine-tuned from LiDAR",
        "livox_e20_on_radar": "LiDAR model fed radar (no training)", "livox_pretrain": "LiDAR model, LiDAR input"}
SENSOR = {m: ("lidar" if m == "livox_pretrain" else "radar") for m in ALL_MODELS}   # what each model sees
CLASSES = ["Vehicle", "Pedestrian", "Bike"]                   # CLASS_NAMES of lidar_only.yaml; id c = CLASSES[c - 1]
IOU_THR = {"Vehicle": 0.5, "Pedestrian": 0.3, "Bike": 0.3}    # same as the official evaluation
MAX_DIST = 70.5                                               # the evaluation ignores everything beyond this
PCR = np.array([0, -76.8, -2, 76.8, 76.8, 4])                 # POINT_CLOUD_RANGE the models were trained with
ON_KAGGLE = os.path.isdir("/kaggle/input")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "frame_inspector_output"
os.makedirs(OUT_DIR, exist_ok=True)
# Where to look for data and predictions: Kaggle's attached datasets, or on a laptop the DSERT-RoLL sample
# folder and the downloaded result.pkl files (add your own paths here).
INPUTS = ["/kaggle/input"] if ON_KAGGLE else \
    [os.path.expanduser("~/dsert-roll-sample"), "dsert_results", "detection/data/dsert-roll"]

# One walk over the attached datasets: drive folders (and their label.pkl), the result.pkl files, the eval logs.
DRIVE = re.compile(r"\d{4}(_\d{2}){5}$")                      # drive folders are named like 2025_04_16_07_04_30
ROOTS, LABELS, PKLS, LOGS = [], {}, [], []
for path, dirs, files in (w for top in INPUTS if os.path.isdir(top) for w in os.walk(top, followlinks=True)):
    dirs[:] = [d for d in dirs if not d.startswith(".")]     # skip e.g. a download tool's .cache
    for d in [d for d in dirs if DRIVE.match(d)]:
        dirs.remove(d)                                        # do not walk into a drive's thousands of frame files
        root = os.path.dirname(path)                          # .../processed_data (path is .../processed_data/<Weather>)
        if root not in ROOTS:
            ROOTS.append(root)
        if os.path.exists(os.path.join(path, d, "label.pkl")):
            LABELS.setdefault(d, os.path.join(path, d, "label.pkl"))
    PKLS += [os.path.join(path, f) for f in files if f.endswith(".pkl")]
    LOGS += [os.path.join(path, f) for f in files if re.match(r"log_(eval|train)_.*\.txt$", f)]

RESULT = {}
for m in ALL_MODELS:
    found = [p for p in PKLS if os.path.basename(p) == m + ".pkl"] + \
            sorted([p for p in PKLS if os.path.basename(p) == "result.pkl" and m in re.split(r"[\\/]", p)],
                   key=lambda p: "epoch_20" not in p)
    if found:
        RESULT[m] = found[0]
MODELS = list(RESULT)
assert MODELS, "no predictions found: attach the result.pkl files (named radar_scratch.pkl, ...)"
assert LABELS, "no label.pkl found: attach the DSERT-RoLL dataset (processed_data/<Weather>/<drive>/label.pkl)"
for m, p in RESULT.items():
    print("predictions %-20s <- %s" % (m, p))
print("%d drives with labels, data folders: %s" % (len(LABELS), ROOTS))

def resolve(rel):
    "A path from label.pkl (relative to processed_data) -> the file in whichever attached dataset has it ('' if none)."
    for r in ROOTS:
        p = os.path.join(r, rel)
        if os.path.exists(p):
            return p
    return ""

def part(key, k):
    "Rows of a flat array that belong to frame k (k = 0..N-1)."
    off = F[key + "_off"]
    return slice(off[k], off[k + 1])

In [ ]:
# Geometry: 3D IoU of rotated boxes [x, y, z, dx, dy, dz, heading] (z = box centre), as the evaluation computes it
def bev_corners(b):
    "Corners of the box seen from above, counter-clockwise."
    c = np.array([[1, 1], [-1, 1], [-1, -1], [1, -1]]) * [b[3] / 2, b[4] / 2]
    ch, sh = np.cos(b[6]), np.sin(b[6])
    return c @ np.array([[ch, sh], [-sh, ch]]) + b[:2]

def clip_polygon(poly, window):
    "The part of convex polygon `poly` inside convex polygon `window` (both counter-clockwise)."
    for a, b in zip(window, np.roll(window, -1, 0)):
        if not len(poly):
            break
        side = lambda p: (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])
        out = []
        for p, q in zip(np.roll(poly, 1, 0), poly):          # edge p -> q
            sp, sq = side(p), side(q)
            if sq >= 0:
                if sp < 0:
                    out.append(p + (q - p) * sp / (sp - sq))
                out.append(q)
            elif sp >= 0:
                out.append(p + (q - p) * sp / (sp - sq))
        poly = np.array(out)
    return poly

def area(poly):
    if len(poly) < 3:
        return 0.0
    x, y = poly[:, 0], poly[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def iou3d(a, b):
    if np.hypot(a[0] - b[0], a[1] - b[1]) > (np.hypot(a[3], a[4]) + np.hypot(b[3], b[4])) / 2:
        return 0.0
    inter = area(clip_polygon(bev_corners(a), bev_corners(b)))
    dz = max(0.0, min(a[2] + a[5] / 2, b[2] + b[5] / 2) - max(a[2] - a[5] / 2, b[2] - b[5] / 2))
    v = inter * dz
    return v / max(a[3] * a[4] * a[5] + b[3] * b[4] * b[5] - v, 1e-9)

def points_in_box(pts, b):
    d = pts[:, :3] - b[:3]
    c, s = np.cos(-b[6]), np.sin(-b[6])
    lx, ly = d[:, 0] * c - d[:, 1] * s, d[:, 0] * s + d[:, 1] * c
    return int(((np.abs(lx) <= b[3] / 2) & (np.abs(ly) <= b[4] / 2) & (np.abs(d[:, 2]) <= b[5] / 2)).sum())

_t = np.array([10, 0, 0, 4, 2, 1.5, 0.])
assert abs(iou3d(_t, _t) - 1) < 1e-9 and abs(iou3d(_t, _t + [1, 0, 0, 0, 0, 0, 0]) - 3 / 5) < 1e-9
assert abs(iou3d(_t, _t + [0, 0, 0, 0, 0, 0, np.pi / 2]) - 1 / 3) < 1e-9   # crossed 4x2 boxes share a 2x2 square

In [ ]:
# Collect every frame the evaluation scored: labelled boxes, radar points, point counts, each model's boxes.
# Same steps as the training / evaluation code (DSERTSingleSensorDataset + dsert_dataset.evaluation):
#   * points: read the .npy, drop empty / NaN points, move radar into LiDAR coordinates through the calibration chain
#     radar -> thermal camera -> LiDAR, keep the point cloud range the models saw
#   * labelled boxes: classes Vehicle / Pedestrian / Bike within 70 m; difficulty 0 = level 1 (scored), else level 2
# The first run reads every radar and LiDAR file once; the result is cached in the output folder.
COUNT_LIDAR = True             # False = do not read the LiDAR files now (faster; LiDAR point counts become -1)
# The cache is named after the drives, prediction files and COUNT_LIDAR it was built from: change any of them
# (e.g. attach more drives) and it is rebuilt instead of silently reusing an old one.
CACHE = os.path.join(OUT_DIR, "frames_cache_%s.npz" % hashlib.md5(repr(
    (sorted(LABELS), sorted((m, os.path.basename(p), os.path.getsize(p)) for m, p in RESULT.items()), COUNT_LIDAR)
).encode()).hexdigest()[:10])
FEATURE_FALLBACKS = ["intensity", "reflectivity", "signal", "doppler"]

def load_points(path, feature):
    "(N, 4) [x, y, z, feature] from a DSERT .npy, as DSERTSingleSensorDataset.load_sensor_points reads it."
    raw = np.load(path)
    if raw.size == 0:
        return np.zeros((0, 4))
    if raw.dtype.names is not None:
        name = next((f for f in [feature] + FEATURE_FALLBACKS if f in raw.dtype.names), "")
        pts = np.stack([raw["x"], raw["y"], raw["z"], raw[name] if name else np.zeros(raw.shape[0])], -1)
    else:
        arr = raw.reshape(-1, raw.shape[-1])
        pts = np.c_[arr[:, :3], arr[:, 3] if arr.shape[1] > 3 else np.zeros(len(arr))]
    pts = pts.astype(np.float64)
    return pts[np.isfinite(pts).all(1) & (np.abs(pts[:, :3]).sum(1) > 0)]

def inverse_T(T):
    T_inv = np.eye(4)
    T_inv[:3, :3] = np.linalg.inv(T[:3, :3])
    T_inv[:3, 3] = -T_inv[:3, :3] @ T[:3, 3]
    return T_inv

def radar_to_lidar(cal):
    "4x4 radar -> Livox transform, as DSERTSingleSensorDataset.sensor_to_livox builds it."
    src = cal.get("Radar", {}) if isinstance(cal.get("Radar"), dict) else {}
    lv = cal.get("Livox", {}) if isinstance(cal.get("Livox"), dict) else {}
    if "Livox" in src:
        return np.asarray(src["Livox"], np.float64)
    if "Radar" in lv:
        return inverse_T(np.asarray(lv["Radar"], np.float64))
    common = [k for k in src if k in lv]
    via = "Thermal_L" if "Thermal_L" in common else common[0]
    return inverse_T(np.asarray(lv[via], np.float64)) @ np.asarray(src[via], np.float64)

def in_range(pts):
    return pts[(pts[:, 0] >= PCR[0]) & (pts[:, 0] < PCR[3]) & (pts[:, 1] >= PCR[1]) & (pts[:, 1] < PCR[4])
               & (pts[:, 2] >= PCR[2]) & (pts[:, 2] < PCR[5])]

def labelled_boxes(info):
    "Boxes the evaluation scores against: (M, 7), class ids (M,), level (M,)."
    a = info.get("annos")
    if not a or len(a["name"]) == 0:
        return np.zeros((0, 7)), np.zeros(0, np.int8), np.zeros(0, np.int8)
    boxes, names = np.asarray(a["gt_boxes_livox"], np.float64), np.asarray(a["name"])
    diff = np.asarray(a.get("difficulty", np.zeros(len(names))), np.int64).reshape(-1)
    diff = np.full(len(names), diff[0]) if diff.size == 1 else diff
    keep = np.isin(names, CLASSES) & (np.hypot(boxes[:, 0], boxes[:, 1]) < MAX_DIST)
    return (boxes[keep, :7], np.array([CLASSES.index(n) + 1 for n in names[keep]], np.int8),
            np.where(np.clip(diff[keep], 0, None) == 0, 1, 2).astype(np.int8))

if os.path.exists(CACHE):
    with np.load(CACHE, allow_pickle=False) as z:
        F = {k: z[k] for k in z.files}
    print("loaded", CACHE, "(delete it to rebuild)")
else:
    preds = {m: {(str(a["sequence_name"]).split("/")[-1], int(a["frame_id"])): a for a in pickle.load(open(p, "rb"))}
             for m, p in RESULT.items()}
    keys = sorted(set().union(*preds.values()))              # (drive, sample) of every frame the evaluation scored
    infos = {}
    for drive in sorted({d for d, _ in keys}):
        if drive in LABELS:
            z = pickle.load(open(LABELS[drive], "rb"))
            for inf in z["info"]:
                inf = dict(inf); inf.update(z["meta"])
                infos[drive, int(inf["sample_idx"])] = inf
    missing = [key for key in keys if key not in infos]
    if missing:
        print("WARNING: %d scored frames have no label.pkl in the attached data (drives: %s); they are skipped"
              % (len(missing), sorted({d for d, _ in missing})))
    keys = [key for key in keys if key in infos]
    assert keys, "none of the predicted frames has a label.pkl in the attached data"

    # Check on the first frame that the files listed in label.pkl are found, before the long pass
    s0 = infos[keys[0]]["sensor"]
    for what, rel in [("radar", s0.get("radar_path", "")), ("LiDAR", s0.get("livox_path", "")),
                      ("camera", s0.get("rgb_left_path", ""))]:
        print("%-6s %s -> %s" % (what, rel, resolve(rel) or "NOT FOUND"))
    assert resolve(s0["radar_path"]), ("radar file not found: label.pkl lists it relative to processed_data, looked in "
                                       "%s. Check that the dataset keeps processed_data/<Weather>/<drive>/..." % ROOTS)
    n_missing = {"radar": 0, "LiDAR": 0, "camera": 0}

    meta = {k: [] for k in ["sample_idx", "sequence", "weather", "light", "has_image", "K", "T_cam", "calib_wh",
                            "img_path", "lidar_path"]}
    flat = {k: [] for k in ["radar_pts", "gt_box", "gt_cls", "gt_level", "gt_radar_pts", "gt_lidar_pts"]}
    for m in MODELS:
        flat.update({"pred_%s_%s" % (m, x): [] for x in ["box", "score", "cls", "pts"]})
    t0 = time.time()
    for n, key in enumerate(keys):
        info = infos[key]
        cal, sensor = info["calibration"], info["sensor"]
        rpath, lpath, ipath = resolve(sensor["radar_path"]), resolve(sensor["livox_path"]), \
            resolve(sensor.get("rgb_left_path", ""))
        for what, p in [("radar", rpath), ("LiDAR", lpath), ("camera", ipath)]:
            n_missing[what] += not p
        rad = load_points(rpath, "doppler") if rpath else np.zeros((0, 4))
        if len(rad):
            rad[:, :3] = rad[:, :3] @ radar_to_lidar(cal)[:3, :3].T + radar_to_lidar(cal)[:3, 3]
        rad = in_range(rad)
        lid = in_range(load_points(lpath, "intensity")) if (COUNT_LIDAR and lpath) else None
        box, cls, level = labelled_boxes(info)
        try:
            K, T_cam = np.asarray(cal["RGB_L"]["intrinsic"], float), np.asarray(cal["Livox"]["RGB_L"], float)
            wh = tuple(cal["RGB_L"].get("shape", (0, 0)))
        except (KeyError, TypeError):
            K, T_cam, wh, ipath = np.zeros((3, 3)), np.zeros((4, 4)), (0, 0), ""
        for k_, v in [("sample_idx", key[1]), ("sequence", str(info.get("sequence_name", key[0]))),
                      ("weather", str(info.get("weather", "?"))), ("light", str(info.get("light", "?"))),
                      ("has_image", bool(ipath)), ("K", K), ("T_cam", T_cam), ("calib_wh", wh),
                      ("img_path", ipath), ("lidar_path", lpath)]:
            meta[k_].append(v)
        flat["radar_pts"].append(rad.astype(np.float32))
        flat["gt_box"].append(box.astype(np.float32)); flat["gt_cls"].append(cls); flat["gt_level"].append(level)
        flat["gt_radar_pts"].append(np.array([points_in_box(rad, b) for b in box], np.int32))
        flat["gt_lidar_pts"].append(np.array([points_in_box(lid, b) if lid is not None else -1 for b in box], np.int32))
        for m in MODELS:
            a = preds[m].get(key)
            ps = np.asarray(a["score"], np.float32).reshape(-1) if a else np.zeros(0, np.float32)
            # boxes_lidar is (n, 7) [x y z dx dy dz heading], or (n, 9) with two velocities: keep the first 7
            pb = np.asarray(a["boxes_lidar"], np.float32).reshape(len(ps), -1)[:, :7] if len(ps) else np.zeros((0, 7), np.float32)
            pc = np.array([CLASSES.index(str(x)) + 1 for x in a["name"]], np.int8) if a else np.zeros(0, np.int8)
            pts = rad if SENSOR[m] == "radar" else lid
            flat["pred_%s_box" % m].append(pb); flat["pred_%s_score" % m].append(ps); flat["pred_%s_cls" % m].append(pc)
            flat["pred_%s_pts" % m].append(np.array([points_in_box(pts, b) if pts is not None else -1 for b in pb],
                                                    np.int32))
        if (n + 1) % 250 == 0 or n + 1 == len(keys):
            print("%d / %d frames, %.0f s" % (n + 1, len(keys), time.time() - t0))

    F = {k: np.array(v) for k, v in meta.items()}
    for k, parts in flat.items():
        F[k] = np.concatenate(parts)
        group = {"radar_pts": "radar", "gt_box": "gt"}.get(k) or (k[:-4] if k.startswith("pred_") and
                                                                   k.endswith("_box") else None)
        if group:
            F[group + "_off"] = np.r_[0, np.cumsum([len(p) for p in parts])]
    F["pred_names"] = np.array(MODELS)
    np.savez(CACHE, **F)
    print("cached in", CACHE)
    if any(n_missing.values()):
        print("WARNING: frames with a file missing in the attached data:", n_missing)

N = len(F["sample_idx"])
MODELS = [str(m) for m in F["pred_names"]]
FR = pd.DataFrame({"drive": [s.split("/")[-1] for s in F["sequence"]], "sample": F["sample_idx"],
                   "weather": F["weather"], "light": F["light"], "has_image": F["has_image"].astype(bool)})
W_ORDER = ["Clear", "Light_Rain", "Heavy_Rain", "Fog", "Light_Snow", "Heavy_Snow"]
WEATHERS = [w for w in W_ORDER if w in set(FR.weather)] + sorted(set(FR.weather) - set(W_ORDER))
print("%d frames (%d with a camera image), %d labelled boxes | models: %s"
      % (N, FR.has_image.sum(), len(F["gt_box"]), MODELS))
display(FR.groupby("weather").agg(frames=("drive", "size"), drives=("drive", "nunique")).reindex(WEATHERS))

In [ ]:
# Match every model's boxes to the labelled boxes, frame by frame (takes a few minutes).
# Highest score first: a box takes the unmatched labelled box of its class with the largest IoU, if that IoU passes the
# class threshold. That is how the evaluation's score sweep sees each box (it uses an optimal assignment, which gives
# the same result except for rare crowded overlaps). Labelled boxes of difficulty level 2 are not scored at level 1:
# boxes matching them are 'IGNORED', like in the evaluation.
GT_ROWS, PRED_ROWS, IOU, STATUS = [], [], {}, {}
for k in range(N):
    gb, gc, gl = F["gt_box"][part("gt", k)], F["gt_cls"][part("gt", k)], F["gt_level"][part("gt", k)]
    gdist = np.hypot(gb[:, 0], gb[:, 1])
    for j in range(len(gb)):
        GT_ROWS.append((k, j, CLASSES[gc[j] - 1], int(gl[j]), gdist[j], int(F["gt_radar_pts"][part("gt", k)][j]),
                        int(F["gt_lidar_pts"][part("gt", k)][j])))
    for m in MODELS:
        s = part("pred_" + m, k)
        pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
        inside = np.hypot(pb[:, 0], pb[:, 1]) < MAX_DIST
        I = np.array([[iou3d(p, g) for g in gb] for p in pb]).reshape(len(pb), len(gb))
        IOU[m, k] = I
        status, match = np.where(inside, "FP", "OUT_OF_RANGE").astype(object), np.full(len(pb), -1)
        taken = np.zeros(len(gb), bool)
        for i in np.argsort(-ps):
            if not inside[i]:
                continue
            cand = [j for j in range(len(gb)) if gc[j] == pc[i] and not taken[j]
                    and I[i, j] >= IOU_THR[CLASSES[gc[j] - 1]]]
            if cand:
                j = max(cand, key=lambda j: I[i, j])
                taken[j], match[i] = True, j
                status[i] = "TP" if gl[j] == 1 else "IGNORED"
        STATUS[m, k] = status
        inside_pts = F["pred_%s_pts" % m][s]
        for i in range(len(pb)):
            PRED_ROWS.append((k, m, i, CLASSES[pc[i] - 1], float(ps[i]), status[i], int(match[i]),
                              float(I[i].max()) if len(gb) else 0.0, int(inside_pts[i])))

GT = pd.DataFrame(GT_ROWS, columns=["k", "j", "cls", "level", "dist", "radar_pts", "lidar_pts"])
PRED = pd.DataFrame(PRED_ROWS, columns=["k", "model", "i", "cls", "score", "status", "matched_gt", "best_iou",
                                        "pts_inside"])
GT = GT.merge(FR[["drive", "sample", "weather", "light"]], left_on="k", right_index=True)
PRED = PRED.merge(FR[["drive", "sample", "weather", "light"]], left_on="k", right_index=True)
print("%d labelled boxes (%d level 1), %d predicted boxes" % (len(GT), (GT.level == 1).sum(), len(PRED)))
display(PRED.groupby(["model", "status"]).size().unstack(fill_value=0))

## 2. The average: AP over all frames (checks this notebook against the official numbers) and the per-frame cut-off

`AP here` should be within about ±0.01 of the official AP; if it is, the per-frame numbers below are counted the
same way as the official score. `cut-off` is the score above which a box counts as "predicted" on the per-frame
pages (the cut-off with the best F1 = balance of precision and recall). Change `CUT` by hand to see the effect.

In [ ]:
def ap_curve(m, cls):
    "Precision and recall after each box of model m and class cls, boxes sorted by score (all frames together)."
    p = PRED[(PRED.model == m) & (PRED.cls == cls) & PRED.status.isin(["TP", "FP"])].sort_values("score", ascending=False)
    n_gt = ((GT.cls == cls) & (GT.level == 1)).sum()
    tp = (p.status == "TP").cumsum().values
    precision, recall = tp / np.arange(1, len(p) + 1), tp / max(n_gt, 1)
    return p.score.values, precision, recall

def average_precision(precision, recall):
    "Area under the precision-recall curve, precision made non-increasing first (standard 'all points' AP)."
    if len(precision) == 0:
        return 0.0
    prec = np.maximum.accumulate(precision[::-1])[::-1]
    return float(np.sum(np.diff(np.r_[0, recall]) * prec))

# Official numbers, if the evaluation logs (log_eval_*.txt / log_train_*.txt) are in an attached dataset
OFFICIAL = {}
for m in MODELS:
    logs = sorted([p for p in LOGS if m in re.split(r"[\\/]", p)], key=os.path.getmtime)   # in a folder named m
    for path in logs:
        text = open(path, errors="replace").read()
        if "==== Overall ====" in text:
            block = text.split("==== Overall ====")[-1]
            for c, tag in [("Vehicle", "VEHICLE"), ("Pedestrian", "PEDESTRIAN"), ("Bike", "CYCLIST")]:
                hit = re.search(r"OBJECT_TYPE_TYPE_%s_LEVEL_1/AP:\s*([0-9.eE-]+)" % tag, block)
                if hit:
                    OFFICIAL[m, c] = float(hit.group(1))

rows, CUT = [], {}
for m in MODELS:
    for c in CLASSES:
        score, precision, recall = ap_curve(m, c)
        f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)
        best = int(np.argmax(f1)) if len(f1) else None
        if c == "Vehicle":
            CUT[m] = float(score[best]) if best is not None else 0.3
        rows.append({"model": m, "class": c, "AP here": round(average_precision(precision, recall), 4),
                     "AP official": OFFICIAL.get((m, c), np.nan),
                     "best-F1 cut-off": round(float(score[best]), 3) if best is not None else np.nan,
                     "recall at cut-off": round(float(recall[best]), 3) if best is not None else np.nan,
                     "precision at cut-off": round(float(precision[best]), 3) if best is not None else np.nan})
display(pd.DataFrame(rows).set_index(["model", "class"]))
print("score cut-off per model for the per-frame pages (vehicle best-F1):", {m: round(v, 3) for m, v in CUT.items()})

fig, ax = plt.subplots(figsize=(8, 6))
for m in MODELS:
    _, precision, recall = ap_curve(m, "Vehicle")
    ax.plot(recall, precision, lw=2, label="%s (AP %.3f)" % (NAME.get(m, m), average_precision(precision, recall)))
ax.set_xlabel("recall (share of labelled vehicles found)"); ax.set_ylabel("precision (share of boxes that are right)")
ax.set_title("Vehicle: AP = area under each curve"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=.3)
ax.legend(); plt.show()

## 3. Every frame's score

One row per frame. For each model: `found/labelled` and the number of false boxes, at that model's cut-off (only
level-1 labelled boxes count, like the official AP). `frame_scores.csv` in the output folder has the same table.
Find a frame by drive and sample number, or sort/filter the table.

In [ ]:
def frame_scores(cut=None):
    cut = cut or CUT
    rows = []
    for m in MODELS:
        p = PRED[(PRED.model == m) & (PRED.score >= cut[m])]
        tp = p[p.status == "TP"].groupby("k").size()
        fp = p[p.status == "FP"].groupby("k").size()
        n = GT[GT.level == 1].groupby("k").size()
        d = pd.DataFrame({"labelled": n, "found": tp, "false": fp}).reindex(range(N)).fillna(0).astype(int)
        d["missed"] = d.labelled - d.found
        d["verdict"] = np.select([(d.missed > 0) & (d.false > 0), d.missed > 0, d.false > 0],
                                 ["BAD+UGLY", "BAD", "UGLY"], "GOOD")
        d["model"] = m
        rows.append(d.reset_index().rename(columns={"index": "k"}))
    return pd.concat(rows, ignore_index=True).merge(FR, left_on="k", right_index=True)

S = frame_scores()
wide = S.assign(cell=S.found.astype(str) + "/" + S.labelled.astype(str) + " +" + S.false.astype(str) + " false"
                ).pivot(index="k", columns="model", values="cell")
wide = FR[["weather", "light", "drive", "sample"]].join(wide)
wide.to_csv(os.path.join(OUT_DIR, "frame_scores.csv"))
print("frame_scores.csv written. Example rows:")
display(wide.head(15))

print("\nVerdicts per model and weather (number of frames):")
display(S.groupby(["model", "weather", "verdict"]).size().unstack(fill_value=0)
          .reindex(columns=["GOOD", "BAD", "UGLY", "BAD+UGLY"], fill_value=0))

# Look up one frame:  wide[(wide.drive == "2025_04_16_07_04_30") & (wide["sample"] == 120)]

## 4. Why each box was missed or false

**Missed labelled box**, first reason that applies:
| reason | meaning |
|---|---|
| `right box, score too low` | the model drew a correct box but was not confident enough (below the cut-off) |
| `no <sensor> points on it` | not a single radar (or LiDAR) point hit the object: the model had nothing to see |
| `box shifted / rotated / wrong size` | a box sits on it but does not overlap enough; the biggest error is named |
| `box went to a neighbour` | the nearby box was used for another labelled object |
| `seen as another class` | a box of the wrong class sits on it |
| `only a weak guess nearby` | something below the cut-off was predicted near it |
| `nothing predicted near it` | the model saw nothing there (points present, but no box within 4 m) |

**False box**: `duplicate` (a second box on an already-found object), `misplaced box on a real object`,
`ghost on clutter` (sensor points inside, but no labelled object; check the camera: it may be an unlabelled car,
a pole, a wall...) or `ghost in empty space` (no points inside at all).

In [ ]:
def miss_reason(m, g, cut):
    "Why model m missed labelled box g (a row of GT) at score cut-off `cut`: (reason, IoU of the closest box)."
    k, j = g.k, g.j
    gb = F["gt_box"][part("gt", k)][j]
    s = part("pred_" + m, k)
    pb, ps, pc = F["pred_%s_box" % m][s], F["pred_%s_score" % m][s], F["pred_%s_cls" % m][s]
    I = IOU[m, k][:, j] if len(pb) else np.zeros(0)
    same = pc == CLASSES.index(g.cls) + 1
    near = np.hypot(pb[:, 0] - gb[0], pb[:, 1] - gb[1]) < 4 if len(pb) else np.zeros(0, bool)
    pts = g.radar_pts if SENSOR[m] == "radar" else g.lidar_pts
    if (same & (I >= IOU_THR[g.cls]) & (ps < cut)).any():
        return "right box, score too low", float(I[same & (ps < cut)].max())
    if pts == 0:
        return "no %s points on it" % SENSOR[m], np.nan
    cand = np.flatnonzero(same & near & (ps >= cut))
    if len(cand):
        i = cand[np.argmax(I[cand])]
        if STATUS[m, k][i] in ("TP", "IGNORED") and I[i] < IOU_THR[g.cls]:
            return "box went to a neighbour", float(I[i])
        shift = np.hypot(*(pb[i][:2] - gb[:2]))
        turn = np.degrees(abs((pb[i][6] - gb[6] + np.pi / 2) % np.pi - np.pi / 2))
        size = max(abs(pb[i][3] - gb[3]) / gb[3], abs(pb[i][4] - gb[4]) / gb[4])
        worst = max([(shift / 1.0, "box shifted %.1f m" % shift), (turn / 20, "box rotated %.0f deg" % turn),
                     (size / 0.25, "box wrong size (%.0f%%)" % (100 * size))])
        return worst[1], float(I[i])
    if ((~same) & (I >= 0.3) & (ps >= cut)).any():
        return "seen as another class", np.nan
    if (near & (ps < cut)).any():
        return "only a weak guess nearby", float(I[near].max())
    return "nothing predicted near it", float(I.max()) if len(I) else 0.0

def false_reason(row):
    "Why a false box (row of PRED with status FP) is false."
    I = IOU[row.model, row.k][row.i]
    same = F["gt_cls"][part("gt", row.k)] == CLASSES.index(row.cls) + 1
    if (same & (I >= IOU_THR[row.cls])).any():      # right place, but that object was already taken
        return "duplicate"
    if row.best_iou >= 0.1:
        return "misplaced box on a real object"
    return "ghost on clutter" if row.pts_inside > 0 else "ghost in empty space"

def explain(cut=None):
    cut = cut or CUT
    miss = []
    for m in MODELS:
        p = PRED[(PRED.model == m) & (PRED.score >= cut[m]) & PRED.status.isin(["TP", "IGNORED"])]
        found = set(zip(p.k, p.matched_gt))
        for g in GT[GT.level == 1].itertuples():
            if (g.k, g.j) not in found:
                r, iou = miss_reason(m, g, cut[m])
                miss.append((m, g.k, g.j, r, re.sub(r" [0-9.]+ ?(m|deg)$| \([0-9]+%\)$", "", r), iou))
    MISS = pd.DataFrame(miss, columns=["model", "k", "j", "reason", "reason_group", "iou"]).merge(
        GT, on=["k", "j"])
    f = PRED[PRED.status.eq("FP") & (PRED.score >= PRED.model.map(cut))].copy()
    f["reason_group"] = [false_reason(r) for r in f.itertuples()]
    return MISS, f

MISS, FALSE = explain()
order_m = ["no radar points on it", "no lidar points on it", "nothing predicted near it", "only a weak guess nearby",
           "right box, score too low", "box shifted", "box rotated", "box wrong size", "box went to a neighbour",
           "seen as another class"]
order_f = ["ghost in empty space", "ghost on clutter", "misplaced box on a real object", "duplicate"]
tab_m = MISS.groupby(["model", "reason_group"]).size().unstack(fill_value=0)
tab_f = FALSE.groupby(["model", "reason_group"]).size().unstack(fill_value=0)
print("MISSED labelled boxes, by reason (all classes, level 1):")
display(tab_m.reindex(columns=[c for c in order_m if c in tab_m.columns]))
print("FALSE boxes, by reason:")
display(tab_f.reindex(columns=[c for c in order_f if c in tab_f.columns]))

REASON_COLOR = dict(zip(order_m + order_f, list(plt.cm.tab10.colors) + list(plt.cm.Set2.colors)))
fig, axes = plt.subplots(1, 2, figsize=(22, 0.9 * len(MODELS) + 3))
for ax, tab, order, title in [(axes[0], tab_m, order_m, "missed labelled boxes"), (axes[1], tab_f, order_f, "false boxes")]:
    cols = [c for c in order if c in tab.columns]
    t = tab.reindex(index=MODELS, columns=cols).fillna(0)
    left = np.zeros(len(t))
    for c in cols:
        ax.barh([NAME.get(m, m) for m in t.index], t[c], left=left, color=REASON_COLOR[c], label=c)
        left += t[c].values
    ax.set_title(title + " per model, by reason (number of boxes)", fontsize=13)
    ax.legend(fontsize=10, bbox_to_anchor=(0, -0.1), loc="upper left", ncol=3)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("The same per weather (missed boxes, share of each reason):")
display((MISS.groupby(["model", "weather", "reason_group"]).size().unstack(fill_value=0)
         .pipe(lambda t: t.div(t.sum(1), axis=0) * 100).round(0)))

## 4b. Recall by angle: do the radar models lose objects at the sides?

**Angle** = direction of a labelled box's centre seen from the Livox: 0° = straight ahead, + = left, − = right.
The sensors see different widths (DSERT-RoLL sensor table): the **radar** (RETINA-4FN) about **±50°**, the **Livox**
about **±60°**, while the models' point range lets boxes reach ±90°. If the radar's narrower view costs detections,
then beyond ~50°:
* the share of labelled boxes with **no radar point** on them goes up, and
* the **recall** (share of labelled boxes found, at each model's cut-off from section 2) of the radar models drops
  while the LiDAR model's holds up to ~60°.

Wide-angle objects are mostly close by, and close objects are easier, so the second table splits by distance too.
The last tables say *why* the radar models miss at each angle (section 4's reasons) and where their false boxes sit.
Small bins (few boxes) are noisy: the `boxes` column says how many there are.

In [ ]:
ANGLE_BINS = [0, 10, 20, 30, 40, 50, 60, 90]        # |angle| in degrees
ANGLE_CLASSES = CLASSES                              # e.g. ["Vehicle"] for cars only
DIST_BINS = [0, 20, 40, 70.5]
RADAR_HALF_FOV, LIDAR_HALF_FOV = 50, 60              # RETINA-4FN 100 deg wide, Livox HAP 120 deg wide
A_MODELS = [m for m in ["radar_scratch", "radar_ft_from_livox", "livox_e20_on_radar", "livox_pretrain"] if m in MODELS]
bin_names = ["%d-%d°" % (a, b) for a, b in zip(ANGLE_BINS[:-1], ANGLE_BINS[1:])]

def box_angle(boxes):
    return np.degrees(np.arctan2(boxes[:, 1], boxes[:, 0]))

# Angle and distance of every labelled box (GT row k, j = box j of frame k)
gb_all = F["gt_box"][F["gt_off"][GT.k.values] + GT.j.values]
GT["angle"] = box_angle(gb_all)
G = GT[(GT.level == 1) & GT.cls.isin(ANGLE_CLASSES)].copy()
G["angle_bin"] = pd.cut(G.angle.abs(), ANGLE_BINS, right=False, labels=bin_names)
G["dist_bin"] = pd.cut(G.dist, DIST_BINS, right=False, labels=["%d-%dm" % (a, b) for a, b in zip(DIST_BINS[:-1], [20, 40, 70])])

def found_at_cut(m, cut=None):
    "True for each row of G that model m found with a box above its cut-off."
    cut = cut or CUT
    p = PRED[(PRED.model == m) & (PRED.score >= cut[m]) & PRED.status.isin(["TP", "IGNORED"])]
    found = set(zip(p.k, p.matched_gt))
    return np.array([(k, j) in found for k, j in zip(G.k, G.j)])

for m in A_MODELS:
    G["found_" + m] = found_at_cut(m)

# 1. Recall per angle bin, next to how many boxes have any radar / LiDAR point on them
t = G.groupby("angle_bin", observed=False).agg(boxes=("k", "size"),
                                               with_radar_pts=("radar_pts", lambda x: (x > 0).mean()))
if (G.lidar_pts >= 0).any():
    t["with_lidar_pts"] = G.groupby("angle_bin", observed=False).lidar_pts.apply(lambda x: (x > 0).mean())
for m in A_MODELS:
    t["recall " + m] = G.groupby("angle_bin", observed=False)["found_" + m].mean()
if {"radar_scratch", "radar_ft_from_livox"} <= set(A_MODELS):
    t["ft - scratch"] = t["recall radar_ft_from_livox"] - t["recall radar_scratch"]
print("Labelled boxes (%s, level 1) by |angle|: share with points on them, and recall at each model's cut-off"
      % ", ".join(ANGLE_CLASSES))
display(t.style.format({c: "{:.0%}" for c in t.columns if c != "boxes"} | {"ft - scratch": "{:+.0%}"}))

fig, ax = plt.subplots(figsize=(10, 5.5))
mid = (np.array(ANGLE_BINS[:-1]) + np.array(ANGLE_BINS[1:])) / 2
for m in A_MODELS:
    ax.plot(mid, 100 * t["recall " + m].values, "-o", lw=2, label="recall: " + NAME.get(m, m))
ax.plot(mid, 100 * t.with_radar_pts.values, "k--", lw=1.5, label="boxes with >= 1 radar point")
if "with_lidar_pts" in t:
    ax.plot(mid, 100 * t.with_lidar_pts.values, ":", c="0.4", lw=1.5, label="boxes with >= 1 LiDAR point")
for x, name in [(RADAR_HALF_FOV, "radar edge"), (LIDAR_HALF_FOV, "Livox edge")]:
    ax.axvline(x, c="0.6", lw=1)
    ax.text(x + 0.5, 101, name, fontsize=9, color="0.4", va="bottom")
for x, n in zip(mid, t.boxes.values):
    ax.text(x, -9, "n=%d" % n, ha="center", fontsize=8, color="0.4")
ax.set_xlabel("|angle| of the labelled box from straight ahead (degrees)"); ax.set_ylabel("%")
ax.set_ylim(-12, 108); ax.set_xlim(0, 90); ax.grid(alpha=.3)
ax.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
ax.set_title("Recall by angle (%s)" % ", ".join(ANGLE_CLASSES)); plt.show()

# 2. The same split by distance (wide angles are mostly close, which is easier)
print("Recall by |angle| and distance, 'recall (boxes)':")
tabs = {}
for m in A_MODELS:
    g = G.groupby(["dist_bin", "angle_bin"], observed=False)["found_" + m].agg(["mean", "size"])
    cell = g.apply(lambda r: "%.0f%% (%d)" % (100 * r["mean"], r["size"]) if r["size"] else "-", axis=1)
    tabs[NAME.get(m, m)] = cell.unstack("angle_bin").reindex(columns=bin_names)
display(pd.concat(tabs, names=["model", "distance"]))

# 3. Why the radar models miss, per angle bin (share of each section-4 reason)
M = MISS.merge(G[["k", "j", "angle_bin"]], on=["k", "j"])
for m in [m for m in A_MODELS if SENSOR[m] == "radar"]:
    tab = M[M.model == m].groupby(["angle_bin", "reason_group"], observed=False).size().unstack(fill_value=0)
    tab = tab.reindex(columns=[c for c in order_m if c in tab.columns])
    print("\n%s: why labelled boxes were missed, per |angle| (share of the misses in that bin, number of misses)"
          % NAME.get(m, m))
    display(tab.div(tab.sum(1).replace(0, np.nan), axis=0).mul(100).round(0).assign(misses=tab.sum(1)))

# 4. Where the false boxes are: per angle bin, false boxes per 100 frames (angle of the predicted box)
fa = []
for r in FALSE.itertuples():
    b = F["pred_%s_box" % r.model][F["pred_%s_off" % r.model][r.k] + r.i]
    fa.append(abs(box_angle(b[None])[0]))
FALSE["angle_bin"] = pd.cut(fa, ANGLE_BINS, right=False, labels=bin_names) if len(fa) else []
print("\nFalse boxes above the cut-off per 100 frames, by |angle| of the false box:")
display((FALSE.groupby(["model", "angle_bin"], observed=False).size().unstack(fill_value=0) * 100 / N).round(1)
        .reindex(A_MODELS))

# 5. Frames to look at: wide-angle boxes the LiDAR model found but the fine-tuned radar model missed
if {"radar_ft_from_livox", "livox_pretrain"} <= set(A_MODELS):
    WIDE = G[(G.angle.abs() >= RADAR_HALF_FOV) & G.found_livox_pretrain & ~G.found_radar_ft_from_livox]
    print("\n%d wide-angle (>= %d deg) boxes found by the LiDAR model but missed by the fine-tuned radar model."
          % (len(WIDE), RADAR_HALF_FOV), "Draw one after running section 5 with show(k), e.g. k in",
          [int(k) for k in sorted(WIDE.k.unique())[:10]])

## 5. Look at frames

`show(frame)` draws one frame for the given models side by side: **camera image** on top (radar points as big dots
coloured by distance, LiDAR points as small grey dots, boxes drawn in 3D) and the **top view** below. Every object
gets a number; the list at the bottom says what happened to it.

Colours: **green** labelled box = found by that model, **red** = missed, **blue** = the model's correct box,
**magenta** = false box, grey dashed = box below the cut-off (ignored).

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
CMAP = plt.get_cmap("turbo")

def corners3d(b):
    c2 = bev_corners(b)
    return np.r_[np.c_[c2, np.full(4, b[2] - b[5] / 2)], np.c_[c2, np.full(4, b[2] + b[5] / 2)]]

def camera(k, max_width=1600):
    "The frame's colour image (shrunk to max_width) and the camera matrix for that size; (None, None) if missing."
    if not F["has_image"][k]:
        return None, None
    img = Image.open(str(F["img_path"][k])).convert("RGB")
    cw, ch = F["calib_wh"][k] if F["calib_wh"][k].min() > 0 else img.size   # size the calibration was made for
    if img.size[0] > max_width:
        img = img.resize((max_width, round(img.size[1] * max_width / img.size[0])))
    K = F["K"][k].copy()
    K[0] *= img.size[0] / cw
    K[1] *= img.size[1] / ch
    return img, (K, F["T_cam"][k])

def lidar_points(k, n=30000):
    "The frame's LiDAR points (a random n of them), read from the dataset when the frame is drawn."
    p = str(F["lidar_path"][k])
    pts = in_range(load_points(p, "intensity")) if p else np.zeros((0, 4))
    return pts if len(pts) <= n else pts[np.random.default_rng(0).choice(len(pts), n, replace=False)]

def to_image(xyz, cam):
    "LiDAR-frame points -> pixel (u, v) and depth in front of the camera; cam = (K, LiDAR->camera transform)."
    K, T = cam
    p = np.c_[xyz, np.ones(len(xyz))] @ T.T
    z = p[:, 2]
    uv = (p[:, :3] @ K.T)
    with np.errstate(divide="ignore", invalid="ignore"):
        return uv[:, 0] / z, uv[:, 1] / z, z

def draw_box(ax_cam, ax_bev, cam, b, color, lw, ls="-", label=None):
    x, y = np.r_[bev_corners(b), bev_corners(b)[:1]].T
    ax_bev.plot(x, y, color=color, lw=lw, ls=ls)
    if label:
        ax_bev.text(b[0], b[1] + b[4] / 2 + 0.6, label, color=color, fontsize=12, ha="center", fontweight="bold",
                    clip_on=True)
    if ax_cam is not None and cam is not None:
        u, v, z = to_image(corners3d(b), cam)
        if (z > 0.5).all():
            for a, c in EDGES:
                ax_cam.plot(u[[a, c]], v[[a, c]], color=color, lw=lw, ls=ls)
            if label:
                ax_cam.text(u[4:].mean(), v[4:].min() - 4, label, color=color, fontsize=12, ha="center",
                            fontweight="bold", bbox=dict(fc="white", alpha=.6, lw=0, pad=1), clip_on=True)

def object_lines(m, k):
    "One text line per labelled object and per false box: what happened and why."
    lines = []
    g = GT[GT.k == k].sort_values("j")
    p = PRED[(PRED.model == m) & (PRED.k == k) & (PRED.score >= CUT[m])]
    hit = {r.matched_gt: (r.i, r.score) for r in p[p.status.isin(["TP", "IGNORED"])].itertuples()}
    miss = MISS[(MISS.model == m) & (MISS.k == k)].set_index("j")
    for r in g.itertuples():
        head = "%d %-7s %2.0fm %3d radar/%4d LiDAR pts" % (r.j + 1, r.cls[:7], r.dist, r.radar_pts, r.lidar_pts)
        if r.j in hit:
            i, score = hit[r.j]
            lines.append(("%s  FOUND (IoU %.2f, score %.2f)" % (head, IOU[m, k][i, r.j], score), "green"))
        elif r.j in miss.index:
            mm = miss.loc[r.j]
            lines.append(("%s  MISSED: %s" % (head, mm.reason) + ("" if pd.isna(mm.iou) else ", IoU %.2f" % mm.iou),
                          "red"))
        else:
            lines.append(("%s  (level 2, not scored)" % head, "gray"))
    fp = FALSE[(FALSE.model == m) & (FALSE.k == k)].sort_values("score", ascending=False)
    for n, r in enumerate(fp.itertuples()):
        lines.append(("%s FALSE %-7s score %.2f: %s, %d %s pts inside" % (
            chr(97 + n % 26), r.cls[:7], r.score, r.reason_group, r.pts_inside, SENSOR[m]), "magenta"))
    return lines

def show(frame=None, sample=None, models=("radar_scratch", "radar_ft_from_livox"), lidar_on_camera=True, title=""):
    "show(k) with k = row of FR (0..N-1), or show(drive_name, sample_number)."
    k = int(FR.index[(FR.drive == frame) & (FR["sample"] == sample)][0]) if sample is not None else int(frame)
    models = [m for m in models if m in MODELS]
    gb = F["gt_box"][part("gt", k)]
    rad = F["radar_pts"][part("radar", k)]
    lid = lidar_points(k)
    img, cam = camera(k)
    n_lines = max(len(object_lines(m, k)) for m in models)
    hr = [1.0 if img is None else img.size[1] / img.size[0] * 1.0, 0.8, 0.045 * max(n_lines, 2) + 0.05]
    fig, axes = plt.subplots(3, len(models), figsize=(13 * len(models), 13 * sum(hr)), squeeze=False,
                             gridspec_kw={"height_ratios": hr})
    kept = [F["pred_%s_box" % m][part("pred_" + m, k)][F["pred_%s_score" % m][part("pred_" + m, k)] >= CUT[m]]
            for m in models]
    xy = np.r_[gb[:, :2].reshape(-1, 2), *[b[:, :2] for b in kept], [[5, -10], [40, 10]]]
    xy = xy[np.hypot(xy[:, 0], xy[:, 1]) < MAX_DIST]
    lim = (max(0, xy[:, 0].min() - 6), min(77, xy[:, 0].max() + 6), xy[:, 1].min() - 6, xy[:, 1].max() + 6)
    for col, m in enumerate(models):
        ax_cam, ax_bev, ax_txt = axes[0, col], axes[1, col], axes[2, col]
        if img is not None:
            ax_cam.imshow(img)
            W, H = img.size
            if lidar_on_camera and len(lid):
                u, v, z = to_image(lid[:, :3], cam)
                ok = (z > 0.5) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
                ax_cam.scatter(u[ok], v[ok], s=1, c="0.85", alpha=.5)
            if len(rad):
                u, v, z = to_image(rad[:, :3], cam)
                ok = (z > 0.5) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
                ax_cam.scatter(u[ok], v[ok], s=40, c=CMAP(np.clip(z[ok] / 70, 0, 1)), edgecolors="k", linewidths=.7)
            ax_cam.set_xlim(0, W); ax_cam.set_ylim(H, 0)
        else:
            ax_cam.text(.5, .5, "no camera image for this frame", ha="center", transform=ax_cam.transAxes)
        ax_cam.axis("off")
        ax_bev.scatter(lid[:, 0], lid[:, 1], s=1, c="0.8")
        ax_bev.scatter(rad[:, 0], rad[:, 1], s=25, c=CMAP(np.clip(np.hypot(rad[:, 0], rad[:, 1]) / 70, 0, 1)),
                       edgecolors="k", linewidths=.6)
        p = PRED[(PRED.model == m) & (PRED.k == k)]
        s = part("pred_" + m, k)
        pb = F["pred_%s_box" % m][s]
        fp_names = {r.i: chr(97 + n % 26) for n, r in
                    enumerate(FALSE[(FALSE.model == m) & (FALSE.k == k)].sort_values("score", ascending=False).itertuples())}
        for r in p.itertuples():
            if r.score < CUT[m]:
                draw_box(None, ax_bev, cam, pb[r.i], "0.6", 0.8, "--")
            elif r.status in ("TP", "IGNORED"):
                draw_box(ax_cam, ax_bev, cam, pb[r.i], "tab:blue", 1.8)
            elif r.status == "FP":
                draw_box(ax_cam, ax_bev, cam, pb[r.i], "magenta", 1.8, label=fp_names.get(r.i))
        found = set(p[(p.score >= CUT[m]) & p.status.isin(["TP", "IGNORED"])].matched_gt)
        for j, b in enumerate(gb):
            draw_box(ax_cam, ax_bev, cam, b, "limegreen" if j in found else "red", 3, label=str(j + 1))
        ax_bev.set_xlim(lim[0], lim[1]); ax_bev.set_ylim(lim[2], lim[3]); ax_bev.set_aspect("equal")
        ax_bev.set_xlabel("x: metres ahead"); ax_bev.set_ylabel("y: metres to the left"); ax_bev.grid(alpha=.3)
        sc = S[(S.model == m) & (S.k == k)].iloc[0]
        ax_cam.set_title("%s\nfound %d of %d, %d false box(es)  ->  %s    (score cut-off %.2f)"
                         % (NAME.get(m, m), sc.found, sc.labelled, sc.false, sc.verdict, CUT[m]), fontsize=15)
        ax_txt.axis("off")
        for n, (line, color) in enumerate(object_lines(m, k)):
            ax_txt.text(0, 1 - n / max(n_lines, 2), line, color=color, fontsize=10.5, family="monospace",
                        va="top", transform=ax_txt.transAxes, clip_on=True)
    fig.suptitle("%sdrive %s | sample %s | %s / %s | %d radar points in the frame"
                 % (title + "  |  " if title else "", FR.drive[k], FR["sample"][k], FR.weather[k], FR.light[k],
                    len(rad)), fontsize=18)
    plt.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

show(int(S[S.labelled > 0].k.iloc[len(S[S.labelled > 0]) // 2]))

## 6. GOOD, BAD and UGLY frames, per weather

For one model (`MODEL`), the frames it handles best and worst in each weather, each drawn next to the other radar
model so you can see what differs. Picks are spread over different drives.

In [ ]:
MODEL = "radar_ft_from_livox"          # whose verdict picks the frames
COMPARE = ("radar_scratch", "radar_ft_from_livox")
N_EACH = 2                              # frames per verdict and weather
SHOW_WEATHERS = WEATHERS                # or e.g. ["Fog", "Heavy_Rain"]

def pick(model, verdict, weather, n):
    d = S[(S.model == model) & (S.weather == weather) & (S.labelled > 0) & FR.has_image.reindex(S.k).values]
    if verdict == "GOOD":
        d = d[d.verdict == "GOOD"].sort_values("labelled", ascending=False)
    elif verdict == "BAD":
        d = d[d.missed > 0].sort_values(["missed", "found"], ascending=[False, True])
    else:
        d = d[d.false > 0].sort_values("false", ascending=False)
    first = d.drop_duplicates("drive")                     # one per drive first, then the rest
    return list(pd.concat([first, d.drop(first.index)]).k.head(n))

for w in SHOW_WEATHERS:
    for verdict in ["GOOD", "BAD", "UGLY"]:
        for k in pick(MODEL, verdict, w, N_EACH):
            show(k, models=COMPARE, title="%s for %s" % (verdict, NAME[MODEL]))

## 7. Where the LiDAR pre-training made the difference

Frames where the fine-tuned radar model finds the most labelled objects that radar-from-scratch misses (and the
other way round), per weather. The object list under each picture says exactly what each model did.

In [ ]:
A, B = "radar_scratch", "radar_ft_from_livox"
N_EACH = 2
d = S[S.model == A][["k", "found", "false"]].merge(S[S.model == B][["k", "found", "false"]], on="k", suffixes=("_A", "_B"))
d["gain"] = d.found_B - d.found_A
d = d.merge(FR, left_on="k", right_index=True)
for w in WEATHERS:
    x = d[(d.weather == w) & d.has_image]
    for label, sel in [("fine-tuned finds more", x[x.gain > 0].sort_values("gain", ascending=False)),
                       ("scratch finds more", x[x.gain < 0].sort_values("gain"))]:
        first = sel.drop_duplicates("drive")
        for k in pd.concat([first, sel.drop(first.index)]).k.head(N_EACH):
            show(k, models=(A, B), title="%s: %s" % (w, label))

## 8. Any frame, or a whole drive

`show(drive, sample)` for one frame. `save_drive` writes one picture per frame of a drive into the output folder
(and a zip) so you can flip through it like a video.

In [ ]:
# One frame: drive name + sample number (from frame_scores.csv or the pictures above)
show(FR.drive.iloc[0], int(FR["sample"].iloc[0]), models=("radar_scratch", "radar_ft_from_livox", "livox_pretrain"))

def save_drive(drive, every=1, models=("radar_scratch", "radar_ft_from_livox")):
    out = os.path.join(OUT_DIR, "drive_" + drive)
    os.makedirs(out, exist_ok=True)
    keep_show = plt.show
    ks = list(FR.index[FR.drive == drive])[::every]
    for n, k in enumerate(ks):
        plt.show = lambda: (plt.savefig(os.path.join(out, "%05d.jpg" % FR["sample"][k]), dpi=50), plt.close("all"))
        show(k, models=models)
    plt.show = keep_show
    with zipfile.ZipFile(out + ".zip", "w") as zf:
        for f in sorted(os.listdir(out)):
            zf.write(os.path.join(out, f), f)
    print("%d pictures in %s (+ .zip)" % (len(ks), out))

# save_drive("2025_04_16_07_04_30", every=2)